# 1.4. Mô hình Lưu trữ 5 Bảng và Kiểm chuẩn Toàn cục Ngoài Bộ nhớ

## 1. Thiết kế lưu trữ kép (Working Layer & Course Export) và kiểm chuẩn SQLite DAG

Hệ thống tách biệt tầng quản trị vận hành (`working`: giữ trọn lịch sử `versions`, trạng thái quan sát và `976.351` trích dẫn) với **5 bảng Parquet chuẩn hóa xuyên suốt Milestone 1 & Milestone 2** (`papers`, `references`, `elements`, `hierarchy`, `bibtex`). Toàn bộ `25.000` bài báo / `41.498` phiên bản được kiểm tra đồng thời bởi `src/parser/validation.py` bằng **chỉ mục SQLite trên đĩa (out-of-core)** nhằm xác minh khóa chính, khóa ngoại xuyên bảng, tính liên thông 1 gốc `document` và tính phi chu trình (DAG) cho từng đồ thị `(arxiv_id, version)` với bộ nhớ RAM phẳng.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
for name, rows in m1["schemas"].items():
    df_sc = pd.DataFrame(rows).rename(columns={"name": "Tên cột", "type": "Kiểu dữ liệu PyArrow", "nullable": "Cho phép Null"})
    display(style_table(df_sc, caption=f"Schema chuẩn bảng `{name}`."))

row_summary = pd.DataFrame([
    {"Giai đoạn": "M1", "Bảng dữ liệu": "papers", "Số bản ghi": scope["papers"], "Khóa chính & Bất biến kiểm chuẩn": "PK (arxiv_id) • 0 trùng lặp chéo giữa 5 thành viên"},
    {"Giai đoạn": "M1", "Bảng dữ liệu": "references", "Số bản ghi": m1["arxiv_reference_rows"], "Khóa chính & Bất biến kiểm chuẩn": "PK (arxiv_id, ref_arxiv_id) • 0 trích dẫn mồ côi, giữ cả ID <2007"},
    {"Giai đoạn": "M2", "Bảng dữ liệu": "elements", "Số bản ghi": m2["rows"]["elements"], "Khóa chính & Bất biến kiểm chuẩn": "PK (element_id) • Khử trùng SHA-256 xuyên version (-35,04%)"},
    {"Giai đoạn": "M2", "Bảng dữ liệu": "hierarchy", "Số bản ghi": m2["rows"]["hierarchy"], "Khóa chính & Bất biến kiểm chuẩn": "PK (arxiv_id, version, element_id, parent_id) • DAG 1 gốc, 0 chu trình"},
    {"Giai đoạn": "M2", "Bảng dữ liệu": "bibtex", "Số bản ghi": m2["rows"]["bibtex"], "Khóa chính & Bất biến kiểm chuẩn": "PK (arxiv_id, citation_key) • Hợp nhất trường & đổi \\cite{} (-35,83%)"},
])
display(style_table(
    row_summary,
    caption="Bảng 4.1. Tổng hợp quy mô 5 bảng hợp nhất (tổng cộng 39.196.741 bản ghi M1–M2).",
    int_cols=["Số bản ghi"],
    bar_cols={"Số bản ghi": ("var(--ah-bar-fill)", None)},
))

df_val = pd.DataFrame([
    ("Toàn vẹn cấu trúc & khóa ngoại 5 bảng (structural_valid)", str(m2["structural_valid"]), "0 lỗi khóa chính, khóa ngoại, gốc document hay chu trình"),
    ("Độ phủ phiên bản M1 -> M2 (expected vs recorded)", f"{fmt_int(scope['versions'])} / {fmt_int(scope['versions'])}", f"missing_versions = {m2['missing_versions']} (độ phủ 100,00%)"),
    ("Bài báo trùng lặp / Trích dẫn mồ côi sau hợp nhất", f"{m1['duplicate_publications_removed']} / {m1['orphan_references_count']}", "Kiểm chứng bất biến bảo toàn khóa trên toàn bộ 5 phân vùng"),
    ("Số lỗi cấu trúc phát hiện bởi validate_project", str(len(m2["validation_errors"])), "Kiểm định bằng SQLite key-index trên đĩa"),
], columns=["Hạng mục kiểm chuẩn toàn cục", "Kết quả", "Chi tiết xác minh"])
display(style_table(df_val, caption="Bảng 4.2. Kết quả kiểm chuẩn toàn cục `validate_project` trên 25.000 bài báo."))

fig, ax = plt.subplots(figsize=(9.6, 3.3))
y = np.arange(len(row_summary))
cols_t = [BOOK_PALETTE["navy"] if g == "M1" else BOOK_PALETTE["teal"] for g in row_summary["Giai đoạn"]]
ax.barh(y, row_summary["Số bản ghi"], color=cols_t, height=0.56)
ax.set_yticks(y); ax.set_yticklabels(row_summary["Bảng dữ liệu"] + " (" + row_summary["Giai đoạn"] + ")")
ax.invert_yaxis(); ax.set_xscale("log"); ax.set_xlim(1e4, 8e7); ax.grid(axis="x")
ax.set_xlabel("Số bản ghi (thang log10)"); ax.set_title("Hình 4.1. Quy mô 5 bảng dữ liệu chuẩn hóa M1–M2 (39,20 triệu dòng, 0 lỗi cấu trúc)")
for i_t, v_t in enumerate(row_summary["Số bản ghi"]):
    ax.text(v_t * 1.18, i_t, fmt_int(v_t), va="center", fontsize=8.4, fontweight="bold")
plt.tight_layout(); plt.show()

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 1 — Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [Đặc tả chính thức Milestone 2 — Hierarchical Parsing & Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/storage.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/storage.py).
- Mã nguồn triển khai: [`src/crawler/merge_shards.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/merge_shards.py).
- Mã nguồn triển khai: [`src/parser/storage.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/storage.py).
- Mã nguồn triển khai: [`src/parser/validation.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/validation.py).
- Mã nguồn triển khai: [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).